In [3]:
%pip install kagglehub

^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached pycparser-2.22-py3-none-any.whl.metadata (943 bytes)
  Using cached markdown_it_py-3.0.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 3.2/3.2 MB 46.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 36.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)


C:\Users\Jerome Yang\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 93%|█████████▎| 2.12G/2.29G [01:21<00:06, 27.8MB/s]


OSError: [Errno 28] No space left on device

In [ ]:
import os
import torch
import torchvision.transforms as transforms
from torchvision.datasets import DatasetFolder
from torch.utils.data import DataLoader, Dataset
from PIL import Image

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize for ViT
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalize grayscale images
])

# Custom dataset to handle bacterial vs. viral pneumonia
class PneumoniaDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # Mapping labels to integers
        self.class_mapping = {
            "normal": 0,
            "bacteria": 1,
            "virus": 2
        }

        # Read images from "normal" and "pneumonia" folders
        for label in ["normal", "pneumonia"]:
            label_dir = os.path.join(root_dir, label)
            for img_name in os.listdir(label_dir):
                img_path = os.path.join(label_dir, img_name)

                # Determine class from filename
                if label == "normal":
                    class_label = "normal"
                elif "bacteria" in img_name.lower():
                    class_label = "bacteria"
                elif "virus" in img_name.lower():
                    class_label = "virus"
                else:
                    continue  # Skip if unknown type

                self.image_paths.append(img_path)
                self.labels.append(self.class_mapping[class_label])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert("RGB")  # Convert grayscale to RGB
        
        if self.transform:
            image = self.transform(image)

        return image, label

# Load dataset
train_data = PneumoniaDataset(root_dir="data/train", transform=transform)
val_data = PneumoniaDataset(root_dir="data/val", transform=transform)
test_data = PneumoniaDataset(root_dir="data/test", transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
